In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_G.xpt"
csv_file = DATA_DIR / "BPQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 BPQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPD035",
    "BPQ040A",
    "BPQ050A",
    "BPQ057",
    "BPQ056",
    "BPD058",
    "BPQ059",
    "BPQ080",
    "BPQ060",
    "BPQ070",
    "BPQ090D",
    "BPQ100D"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BPQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 15:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,175 rows x 15 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached BPQ_G.xpt:
#
#     tiny artifacts = 0
#
# Therefore NO zero restoration is required.
#
# If your local copy ever shows tiny values,
# STOP rather than replacing them automatically.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BPQ_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total != 0:

    print(
        "\nTrue zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected true numeric zeros "
        "found in BPQ_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No true numeric zeros."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# IMPORTANT:
#
# Always check this BEFORE Int64 conversion.
#
# Attached BPQ_G:
#
#     genuine fractional variables = NONE
#
# If fractions unexpectedly appear,
# STOP instead of rounding them.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# All 15 BPQ_G fields are proven whole-number.
#
# Convert to nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimals "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 15 BPQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 8. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 9. CDC CHECK - BPQ020
# ============================================================

#
# Ever told had high blood pressure
#
# CDC:
#
# 1 Yes          2031
# 2 No           4134
# 7 Refused         0
# 9 Don't know      9
# Missing           1
#

check_frequency(
    "BPQ020",
    {
        1: 2031,
        2: 4134,
        9: 9
    },
    1
)


# ============================================================
# 10. CDC CHECK - BPQ030
# ============================================================

#
# Told had hypertension on 2+ visits
#

check_frequency(
    "BPQ030",
    {
        1: 1619,
        2: 407,
        7: 1,
        9: 4
    },
    4144
)


# ============================================================
# 11. CDC CHECK - BPD035
# ============================================================

#
# Age first told had hypertension
#
# CDC:
#
# 5-79 = 1,936
# 80   = 36
# 777  = 1
# 999  = 54
# Missing = 4,148
#

s = df["BPD035"]


print("\n--- BPD035 ---")


if not (
    int(
        s.between(
            5,
            79
        ).sum()
    )
    == 1936
    and
    int(
        s.eq(80)
        .sum()
    )
    == 36
    and
    int(
        s.eq(777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(999)
        .sum()
    )
    == 54
    and
    int(
        s.isna()
        .sum()
    )
    == 4148
):

    raise ValueError(
        "BPD035 differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected zero found in BPD035."
    )


print(
    "PASS: BPD035 matches CDC."
)


# ============================================================
# 12. CDC CHECK - BPQ040A
# ============================================================

check_frequency(
    "BPQ040A",
    {
        1: 1745,
        2: 283,
        9: 3
    },
    4144
)


# ============================================================
# 13. CDC CHECK - BPQ050A
# ============================================================

check_frequency(
    "BPQ050A",
    {
        1: 1521,
        2: 223,
        9: 1
    },
    4430
)


# ============================================================
# 14. CDC CHECK - BPQ057
# ============================================================

check_frequency(
    "BPQ057",
    {
        1: 982,
        2: 5147,
        7: 1,
        9: 44
    },
    1
)


# ============================================================
# 15. CDC CHECK - BPQ056
# ============================================================

check_frequency(
    "BPQ056",
    {
        1: 1481,
        2: 4690,
        7: 1,
        9: 2
    },
    1
)


# ============================================================
# 16. CDC CHECK - BPD058
# ============================================================

#
# How often checked BP at home last 12 months
#
# CDC:
#
# 1-2192 = 1,474
# 7777   = 0
# 9999   = 7
# Missing = 4,694
#
# IMPORTANT:
#
# Unlike BPQ_F, attached BPQ_G contains
# NO tiny/zero values in BPD058.
#

s = df["BPD058"]


print("\n--- BPD058 ---")


if not (
    int(
        s.between(
            1,
            2192
        ).sum()
    )
    == 1474
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 7
    and
    int(
        s.isna()
        .sum()
    )
    == 4694
):

    raise ValueError(
        "BPD058 differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected zero found in BPD058."
    )


print(
    "PASS: BPD058 matches CDC."
)


# ============================================================
# 17. CDC CHECK - BPQ059
# ============================================================

check_frequency(
    "BPQ059",
    {
        1: 791,
        2: 5379,
        7: 1,
        9: 3
    },
    1
)


# ============================================================
# 18. CDC CHECK - BPQ080
# ============================================================

#
# Doctor told participant cholesterol was high
#

check_frequency(
    "BPQ080",
    {
        1: 1826,
        2: 3696,
        7: 1,
        9: 36
    },
    616
)


# ============================================================
# 19. CDC CHECK - BPQ060
# ============================================================

#
# Ever had blood cholesterol checked
#

check_frequency(
    "BPQ060",
    {
        1: 2264,
        2: 1365,
        7: 1,
        9: 103
    },
    2442
)


# ============================================================
# 20. CDC CHECK - BPQ070
# ============================================================

#
# When cholesterol last checked
#

check_frequency(
    "BPQ070",
    {
        1: 2670,
        2: 662,
        3: 494,
        4: 233,
        9: 28
    },
    2088
)


# ============================================================
# 21. CDC CHECK - BPQ090D
# ============================================================

#
# Told to take prescription medicine
# to lower cholesterol
#

check_frequency(
    "BPQ090D",
    {
        1: 1351,
        2: 2728,
        9: 8
    },
    2088
)


# ============================================================
# 22. CDC CHECK - BPQ100D
# ============================================================

#
# Currently following prescribed medicine advice
#

check_frequency(
    "BPQ100D",
    {
        1: 1058,
        2: 291,
        9: 2
    },
    4824
)


# ============================================================
# 23. INTERNAL LOGIC CHECKS
# ============================================================

#
# Some skip-pattern consistency checks.
# These are not substitutions for the CDC frequency checks;
# they are additional protections against corruption.
#

print("\n--- INTERNAL LOGIC CHECKS ---")


#
# BPQ030 is only asked when BPQ020 = Yes.
#

if not (
    df.loc[
        df["BPQ020"].eq(1),
        "BPQ030"
    ]
    .notna()
    .all()
):

    raise ValueError(
        "BPQ030 missing unexpectedly "
        "among BPQ020=1 participants."
    )


#
# BPQ050A is asked when BPQ040A = Yes.
#

if not (
    df.loc[
        df["BPQ040A"].eq(1),
        "BPQ050A"
    ]
    .notna()
    .all()
):

    raise ValueError(
        "BPQ050A missing unexpectedly "
        "among BPQ040A=1 participants."
    )


#
# BPD058 is asked when BPQ056 = Yes.
#

if not (
    df.loc[
        df["BPQ056"].eq(1),
        "BPD058"
    ]
    .notna()
    .all()
):

    raise ValueError(
        "BPD058 missing unexpectedly "
        "among BPQ056=1 participants."
    )


#
# BPQ100D is asked when BPQ090D = Yes.
#

if not (
    df.loc[
        df["BPQ090D"].eq(1),
        "BPQ100D"
    ]
    .notna()
    .all()
):

    raise ValueError(
        "BPQ100D missing unexpectedly "
        "among BPQ090D=1 participants."
    )


print(
    "PASS: Key questionnaire skip patterns are consistent."
)


# ============================================================
# 24. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# BPQ_G requires ZERO numeric correction.
#
# Therefore every numeric value must remain exactly
# equal to the source XPT.
#
# Only textual CSV representation changes:
#
#     62161.0 -> 62161
#     1.0     -> 1
#     9999.0  -> 9999
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 15 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# No rounding.
# No float_format.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 26. RE-READ CSV
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 27. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 15 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 28. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 29. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 30. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 31. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# BPQ_G has NO genuine decimal variables.
#
# Therefore no populated field should end in ".0".
#
# This is safe for THIS FILE.
#
# DO NOT use this global rule for Dietary or Lab/Exam
# files containing genuine decimal measurements.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "BPQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "True numeric zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 BPQ_G XPT -> CSV
Rows: 6,175
Columns: 15
PASS: Structure = 6,175 rows x 15 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 0
PASS: No tiny XPORT artifacts.
PASS: No true numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
BPQ020: 0 fractional observations
BPQ030: 0 fractional observations
BPD035: 0 fractional observations
BPQ040A: 0 fractional observations
BPQ050A: 0 fractional observations
BPQ057: 0 fractional observations
BPQ056: 0 fractional observations
BPD058: 0 fractional observations
BPQ059: 0 fractional observations
BPQ080: 0 fractional observations
BPQ060: 0 fractional observations
BPQ070: 0 fractional observations
BPQ090D: 0 fractional observations
BPQ100D: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

PASS: All 15 BPQ_G variables converted to nullable Int64 without rounding.
PASS: 